# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The method I choose to use is the "yes/no" with observed label using logistic regression. This label attempts to a answer a question of: Did a declining page recover by the next time period? This fits the refresh/content opportunity scoring lane because it studies the refresh potential of a declining page. To start, logistic regression will be used. The output model's coefficients will provide explanations to important features. PCA will be used if necessary to reduce correlation. Random forest/boost methods will be considered for precision comparison.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Training can only occur on client data with enough history. A cutoff date T will be selected and tuned to train the model using future (after T) recovery data and past (before T) decline data. Clients are to be split into train/test.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# cross validation
# rel sample

In [47]:
date_range = con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()
earliest, latest = date_range["earliest"][0], date_range["latest"][0]

candidate_Ts = pd.date_range(earliest, latest, freq="MS")

scan_results = []
for T_candidate in candidate_Ts:
    T_str = T_candidate.strftime("%Y-%m-%d")
    n = con.sql(f"""
        WITH client_ranges AS (
            SELECT client_hash_id, MIN(report_date) AS min_date, MAX(report_date) AS max_date
            FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
            GROUP BY client_hash_id
        )
        SELECT COUNT(DISTINCT CASE WHEN min_date <= (DATE '{T_str}' - INTERVAL 59 DAY)
                                    AND max_date >= (DATE '{T_str}' + INTERVAL 61 DAY)
                               THEN client_hash_id END) AS n
        FROM client_ranges
    """).df()["n"][0]
    scan_results.append({"T": T_str, "eligible_clients": n})

scan_df = pd.DataFrame(scan_results)
print(scan_df)
print(f"\nBest T: {scan_df.loc[scan_df['eligible_clients'].idxmax(), 'T']} "
      f"with {scan_df['eligible_clients'].max()} eligible clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

             T  eligible_clients
0   2025-02-01                 0
1   2025-03-01                 0
2   2025-04-01                 2
3   2025-05-01                 3
4   2025-06-01                 4
5   2025-07-01                 4
6   2025-08-01                 4
7   2025-09-01                10
8   2025-10-01                16
9   2025-11-01                16
10  2025-12-01                24
11  2026-01-01                32
12  2026-02-01                39
13  2026-03-01                39
14  2026-04-01                38
15  2026-05-01                 0
16  2026-06-01                 0

Best T: 2026-02-01 with 39 eligible clients


In [ ]:
T = "2026-02-01"
WINDOW_DAYS = 30       # comparison window size (last30/prev30 convention)
LABEL_GAP_DAYS = 1     # days between T and the start of the recovery-check window
LABEL_WINDOW_DAYS = 30 # size of the recovery-check window

# --- feature set, as of T only, no label-derived fields ---
content_type_query = f"""
    SELECT content_hash_id, keyword_char_count, url_char_count, content_type,
        search_volume, competition, main_intent, category_count, model_used, char_count,
        DATEDIFF('day', content_updated_date, DATE '{T}') AS days_since_last_update,
        DATEDIFF('day', content_created_date, DATE '{T}') AS days_since_created
    FROM read_parquet('{rel}/dim_content.parquet')
"""
content_dim = con.sql(content_type_query).df()

feature_query = f"""
    SELECT client_hash_id, content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date <= DATE '{T}'
    GROUP BY client_hash_id, content_hash_id
"""
X_raw = con.sql(feature_query).df().merge(content_dim, on="content_hash_id", how="left")

# --- gate: is_declining at T -- window boundaries derived entirely from T + WINDOW_DAYS ---
gate_query = f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {WINDOW_DAYS - 1} DAY) AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                           AND (DATE '{T}' - INTERVAL {WINDOW_DAYS} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 < -20
             THEN 1 ELSE 0 END AS is_declining_at_T
    FROM windowed
"""
gate_df = con.sql(gate_query).df()

# --- recovery label: strictly after T -- also fully derived ---
future_query = f"""
    WITH future AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_T1,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_baseline
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date > DATE '{T}'
          AND report_date <= (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN impr_T1 > impr_baseline THEN 1 ELSE 0 END AS recovered_by_T1
    FROM future
"""
recovery_df = con.sql(future_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [1]:
print(X_raw.isna().sum())

NameError: name 'X_raw' is not defined

In [51]:
#X_raw["gsc_avg_position"] = X_raw["gsc_avg_position"].fillna(0)
X_raw["search_volume"] = X_raw["search_volume"].fillna(0)
X_raw["competition"] = X_raw["competition"].fillna(0)
#X_raw["main_intent"] = X_raw["main_intent"].fillna("no_keyword")
#X_raw["model_used"] = X_raw["model_used"].fillna("human")
X_raw["char_count"] = X_raw["char_count"].fillna(X_raw["char_count"].median())

In [52]:
print(X_raw.isna().sum())

client_hash_id            0
content_hash_id           0
gsc_avg_position          0
gsc_impressions           0
gsc_clicks                0
keyword_char_count        0
url_char_count            0
content_type              0
search_volume             0
competition               0
main_intent               0
category_count            0
model_used                0
char_count                0
days_since_last_update    0
days_since_created        0
dtype: int64


In [53]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

declining_ids = gate_df.loc[gate_df["is_declining_at_T"] == 1, ["client_hash_id", "content_hash_id"]]

data = (X_raw.merge(declining_ids, on=["client_hash_id", "content_hash_id"], how="inner")
             .merge(recovery_df[["client_hash_id", "content_hash_id", "recovered_by_T1"]],
                     on=["client_hash_id", "content_hash_id"], how="inner"))

y = data.pop("recovered_by_T1")
groups = data["client_hash_id"]
X = pd.get_dummies(data.drop(columns=["client_hash_id", "content_hash_id"]), columns=["content_type", "main_intent", "model_used"])

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"train: {len(X_train)} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"test:  {len(X_test)} rows, {groups.iloc[test_idx].nunique()} clients")

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [42]:
numeric_cols = X_train.select_dtypes(include="number").columns
corr_matrix = X_train[numeric_cols].corr().abs()
import numpy as np
np.fill_diagonal(corr_matrix.values, 0)
print(corr_matrix.unstack().sort_values(ascending=False).head(10))

gsc_impressions         gsc_clicks                0.503077
gsc_clicks              gsc_impressions           0.503077
gsc_avg_position        days_since_last_update    0.304284
days_since_last_update  gsc_avg_position          0.304284
gsc_impressions         days_since_last_update    0.207126
days_since_last_update  gsc_impressions           0.207126
keyword_char_count      days_since_created        0.170781
days_since_created      keyword_char_count        0.170781
search_volume           keyword_char_count        0.162428
keyword_char_count      search_volume             0.162428
dtype: float64


In [40]:
import numpy as np
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

logit = LogisticRegression(random_state=42, class_weight="balanced", max_iter=2000).fit(X_train_scaled, y_train)
rf = RandomForestClassifier(max_depth=4, random_state=42, class_weight="balanced").fit(X_train, y_train)

def precision_at_k(scores, labels, k, seed=42):
    rng = np.random.default_rng(seed)
    shuffle = rng.permutation(len(scores))
    order = shuffle[np.argsort(-np.asarray(scores)[shuffle], kind="stable")]
    return np.asarray(labels)[order[:k]].mean()

stayed_broken = 1 - y_test.values

# guard against k being larger than the actual test set -- with few clients, test set may be small
print(f"test set size: {len(y_test)}")

results = []
for name, scores in [
    ("logistic regression (1 - P(recovery))", 1 - logit.predict_proba(X_test_scaled)[:, 1]),  # <-- fixed
    ("random forest (1 - P(recovery))", 1 - rf.predict_proba(X_test)[:, 1]),  # unscaled is correct for RF
]:
    for k in (20, 50, 200):
        k_actual = min(k, len(y_test))
        results.append((name, k, precision_at_k(scores, stayed_broken, k_actual)))

comparison = pd.DataFrame(results, columns=["method", "k", "precision"]).pivot(index="method", columns="k", values="precision")
comparison["base_rate"] = stayed_broken.mean()
print(comparison)

# --- coefficients (logistic regression) ---
coefficients = pd.Series(logit.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)
print("\nLogistic regression coefficients (sign = direction toward recovery):")
print(coefficients.head(10))

# --- feature importances (random forest) ---
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nRandom forest feature importances:")
print(importances.head(10))

test set size: 22816
k                                        20    50    200  base_rate
method                                                             
logistic regression (1 - P(recovery))  0.05  0.28  0.145   0.193811
random forest (1 - P(recovery))        0.00  0.00  0.000   0.193811

Logistic regression coefficients (sign = direction toward recovery):
days_since_created          -0.355457
model_used_gpt-4o-mini       0.207999
main_intent_navigational    -0.205483
model_used_human            -0.117340
gsc_impressions             -0.107982
competition                 -0.100790
main_intent_transactional    0.077674
main_intent_commercial      -0.066717
char_count                  -0.066245
model_used_unknown           0.047836
dtype: float64

Random forest feature importances:
days_since_created           0.388656
competition                  0.089781
model_used_gpt-4o-mini       0.086294
search_volume                0.063205
url_char_count               0.058366
char_count      

In [31]:
from sklearn.metrics import roc_auc_score

# alignment sanity check -- rule out the simplest possible bug first
print("X_test/y_test aligned:", X_test.index.equals(y_test.index))

# AUC for both models -- tells us if RF's signal is weak vs. actively inverted
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
logit_auc = roc_auc_score(y_test, logit.predict_proba(X_test_scaled)[:, 1])
print(f"Random forest AUC: {rf_auc:.3f}")
print(f"Logistic regression AUC: {logit_auc:.3f}")

# how many distinct test clients, and how big is the positive class
print(f"test clients: {groups.iloc[test_idx].nunique()}")
print(f"test set size: {len(y_test)}, recovered rate: {y_test.mean():.3f}")

X_test/y_test aligned: True
Random forest AUC: 0.479
Logistic regression AUC: 0.483
test clients: 1
test set size: 22816, recovered rate: 0.806


In [32]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

gkf = GroupKFold(n_splits=4)
fold_results = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    rf_fold = RandomForestClassifier(max_depth=4, random_state=42, class_weight="balanced").fit(X_tr, y_tr)
    auc_fold = roc_auc_score(y_te, rf_fold.predict_proba(X_te)[:, 1])

    fold_results.append({
        "held_out_client": groups.iloc[test_idx].iloc[0],
        "test_rows": len(y_te),
        "recovered_rate": y_te.mean(),
        "auc": auc_fold,
    })

print(pd.DataFrame(fold_results))

           held_out_client  test_rows  recovered_rate       auc
0  client_73cda7b4e4f265ea     101543        0.464257  0.496384
1  client_9958f0a7ae1df715      22816        0.806189  0.478736
2  client_fef1a8f436438636       5418        0.768918  0.570599
3  client_ff644d8251367cbb         29        0.965517  0.857143


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.